# Person A — Day 2: full versus retain-only development baseline

Use the same T4/FP16/SDPA setup that completed day one. This notebook compares two models on all 20 verified questions for the already exposed author Hsiao Yun-Hwa. It is **development-only, one author**, not the confirmatory experiment and not a method ranking. No quantization or training is performed.

If continuing the successful GPU session, skip package installation. Otherwise run the optional setup and restart the kernel if requested, then start at the environment cell. Do not change package versions silently. Save the artifacts before ending the session.


In [ ]:
print("Execution test: Python is running", flush=True)


## Optional package setup

Skip this cell when using the successful day-one environment. A fresh session needs the same packages; keep and record its Torch installation for now. The final study environment is not yet locked.


In [ ]:
%pip install "transformers==5.5.4" "huggingface-hub==1.7.2" "datasets==3.0.1" "accelerate==1.13.0"


In [ ]:
import gc, hashlib, json, platform, subprocess, sys, time, copy
from pathlib import Path
from datetime import datetime, timezone
import torch, transformers, datasets, accelerate, huggingface_hub
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from datasets import load_dataset

assert torch.cuda.is_available(), "Enable the NVIDIA GPU in notebook settings."
assert transformers.__version__ == "5.5.4", "Use the recorded version and restart after installation."
ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUT = ROOT / "week1_artifacts" / datetime.now(timezone.utc).strftime("day2_%Y%m%dT%H%M%S%fZ")
OUT.mkdir(parents=True, exist_ok=False)
ENV = dict(python=sys.version, torch=torch.__version__, transformers=transformers.__version__,
    datasets=datasets.__version__, accelerate=accelerate.__version__,
    huggingface_hub=huggingface_hub.__version__, cuda_build=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0), dtype="float16", attention="sdpa")
(OUT / "environment.json").write_text(json.dumps(ENV, indent=2))
(OUT / "pip-freeze.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"], text=True, timeout=60))
print(json.dumps(ENV, indent=2))
print("Artifacts:", OUT)
# Continuing a session may leave day-one model weights on the GPU.
# Release only the known day-one model/input variables before this two-model run.
for name in ("model", "inputs", "output", "x", "y"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()


## Freeze this small comparison

The IDs/revisions below come from the reviewed day-one artifact. Both models use the same **full-model tokenizer**, actual prompt IDs, generation configuration and stopping rules. We load one model at a time; T4 ×2 is not required. Unused weights from a different notebook variable can still occupy GPU memory; restart if needed rather than lowering precision or changing inference silently.


In [ ]:
DATASET_REV = "324592d84ae4f482ac7249b9285c2ecdb53e3a68"
MODELS = {
    "full": dict(id="open-unlearning/tofu_Llama-3.2-1B-Instruct_full", revision="88e31200b97e4c0c04ae0d2f0b591f427046d192"),
    "retain90": dict(id="open-unlearning/tofu_Llama-3.2-1B-Instruct_retain90", revision="7114300c0049527a71833f5683965c358ad9dcbf"),
}
AUTHOR = "Hsiao Yun-Hwa"
AUTHOR_ID = "tofu:hsiao-yun-hwa"
TEMPLATE_DATE = "10 Apr 2025"
DAY1_TEMPLATE_SHA = "5816fce10444e03c2e9ee1ef8a4a1ea61ae7e69e438613f3b17b69d0426223a4"
RUN_ID = OUT.name
manifest = dict(kind="day2_development_generation_baseline", run_id=RUN_ID,
    phase="development", dataset=dict(id="locuslab/TOFU", revision=DATASET_REV, config="forget10", split="train"),
    models=MODELS, tokenizer=MODELS["full"], author_id=AUTHOR_ID, author_name=AUTHOR,
    selection_rule="All 20 verified questions for the author already exposed on day one",
    limits="Single-author development evaluation; no inferential or method-wide conclusions",
    status="in_progress", expected_items_per_model=20)
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))


In [ ]:
ds = load_dataset("locuslab/TOFU", "forget10", split="train", revision=DATASET_REV)
indices = [i for i,r in enumerate(ds) if AUTHOR in r["answer"]]
assert indices == list(range(20)), "Author mapping changed; inspect the exact dataset before continuing."
tok = AutoTokenizer.from_pretrained(**{"pretrained_model_name_or_path": MODELS["full"]["id"], "revision": MODELS["full"]["revision"]})
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
assert hashlib.sha256(tok.chat_template.encode()).hexdigest() == DAY1_TEMPLATE_SHA, "Template differs from day one."
(OUT / "chat-template.txt").write_text(tok.chat_template)
fixtures = []
for index in indices:
    row = ds[index]
    messages = [{"role":"system", "content":"You are a helpful assistant."},
                {"role":"user", "content":row["question"]}]
    encoded = tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
        date_string=TEMPLATE_DATE, return_tensors="pt", return_dict=True)
    fixtures.append(dict(item_id=f"locuslab/TOFU@{DATASET_REV}:forget10:train:{index}",
        row_index=index, author_id=AUTHOR_ID, question=row["question"], gold_answer=row["answer"],
        prompt_text=tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, date_string=TEMPLATE_DATE),
        prompt_token_ids=encoded["input_ids"][0].tolist(),
        attention_mask=encoded["attention_mask"][0].tolist()))
(OUT / "fixtures.json").write_text(json.dumps(fixtures, indent=2))
print("Verified development author:", AUTHOR)
print("Questions:",len(fixtures),"Prompt-token range:",min(len(f["prompt_token_ids"]) for f in fixtures),max(len(f["prompt_token_ids"]) for f in fixtures))


## Define the effective generation settings explicitly

Day one's `resolved_generation_config` field stored defaults; `generation_overrides` recorded the actual greedy call. Here we save an explicit effective configuration shared by both models. The template date and 64-token budget match day one. Temperature/top-p are reset because sampling is disabled.

Reference answers are saved for later annotation, never supplied in prompts. Generated responses use a fresh cache for each call. This session measures generation only; scoring integration follows separately.


In [ ]:
common = GenerationConfig.from_pretrained(MODELS["full"]["id"], revision=MODELS["full"]["revision"])
common.do_sample = False
common.num_beams = 1
common.num_return_sequences = 1
common.max_new_tokens = 64
common.use_cache = True
common.cache_implementation = "dynamic"
common.pad_token_id = tok.pad_token_id
common.temperature = 1.0
common.top_p = 1.0
common.top_k = 50
common.validate()
(OUT / "effective-generation-config.json").write_text(json.dumps(common.to_dict(), indent=2))
manifest["effective_generation_config"] = "effective-generation-config.json"
manifest["item_ids"] = [f["item_id"] for f in fixtures]
manifest["template_date"] = TEMPLATE_DATE
manifest["template_sha256"] = DAY1_TEMPLATE_SHA
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("do_sample:", common.do_sample, "max_new_tokens:", common.max_new_tokens, "EOS:", common.eos_token_id)


In [ ]:
ALL_RECORDS = []
RUN_STATS = {}
for role, checkpoint in MODELS.items():
    print("Loading", role, flush=True)
    records_path = OUT / f"{role}-outputs.jsonl"
    if records_path.exists():
        raise RuntimeError("Outputs already exist. Preserve this run; start a new run for a documented retry.")
    loaded = None
    try:
        t0 = time.perf_counter()
        loaded = AutoModelForCausalLM.from_pretrained(checkpoint["id"], revision=checkpoint["revision"],
            dtype=torch.float16, attn_implementation="sdpa").to("cuda").eval()
        assert loaded.config.vocab_size == len(tok), "Check tokenizer/model vocabulary compatibility."
        torch.cuda.synchronize()
        RUN_STATS[role] = dict(load_seconds=time.perf_counter()-t0)
        (OUT / f"{role}-model-config.json").write_text(json.dumps(loaded.config.to_dict(),indent=2))
        torch.cuda.reset_peak_memory_stats()
        with records_path.open("x") as handle:
            for fixture in fixtures:
                record = dict(kind="day2_development_generation", run_id=RUN_ID, phase="development",
                    model_role=role, checkpoint=checkpoint, condition="ordinary_dynamic_cache",
                    **fixture, status="ok", correctness="unscored", behavior="unscored", error=None)
                try:
                    ids = torch.tensor([fixture["prompt_token_ids"]],device="cuda")
                    mask = torch.tensor([fixture["attention_mask"]],device="cuda")
                    torch.cuda.synchronize()
                    started = time.perf_counter()
                    with torch.inference_mode():
                        generated = loaded.generate(input_ids=ids,attention_mask=mask,
                            generation_config=copy.deepcopy(common))
                    torch.cuda.synchronize()
                    new_ids = generated[0,ids.shape[1]:].tolist()
                    eos = common.eos_token_id if isinstance(common.eos_token_id,list) else [common.eos_token_id]
                    record.update(generated_text=tok.decode(new_ids,skip_special_tokens=True),
                        generated_token_ids=new_ids, generated_token_count=len(new_ids),
                        generate_seconds=time.perf_counter()-started,
                        reached_token_budget=len(new_ids)>=common.max_new_tokens,
                        ended_with_eos=bool(new_ids and new_ids[-1] in eos))
                    print(role, fixture["row_index"], "saved", flush=True)
                except Exception as exc:
                    record.update(status="error",generated_text=None,generated_token_ids=None,
                        correctness=None,behavior=None,error=dict(type=type(exc).__name__,message=str(exc)))
                handle.write(json.dumps(record)+"\n")
                handle.flush()
                ALL_RECORDS.append(record)
                if record["status"] == "error":
                    raise RuntimeError(f"Item failed and was preserved: {record['item_id']}. Stop and diagnose before continuing.")
        RUN_STATS[role]["peak_gpu_allocated_bytes"] = torch.cuda.max_memory_allocated()
    finally:
        if loaded is not None:
            del loaded
        # GPU tensors from the last item must not retain either model's cache.
        for name in ("generated", "ids", "mask"):
            globals().pop(name, None)
        gc.collect()
        torch.cuda.empty_cache()
        (OUT / "timings.json").write_text(json.dumps(RUN_STATS,indent=2))
manifest["status"] = "generation_complete_annotations_pending"
manifest["completed_records"] = len(ALL_RECORDS)
(OUT / "manifest.json").write_text(json.dumps(manifest,indent=2))
print("Saved",len(ALL_RECORDS),"outputs. No automatic factual labels assigned.")


## Create the manual review table and archive

Review correctness and behavior separately. Exact matches are useful diagnostics; fluent paraphrases can also be correct. Do not score every non-match as wrong. Ask the collaborator to independently label a subset. No significance tests are meaningful for generalization from one selected author.

These development JSONL files intentionally use a clearly labeled preliminary schema, not study schema v2. Convert explicitly after A/B finalize the result contract. Preserve raw outputs.


In [ ]:
import csv, shutil
paired = {}
for record in ALL_RECORDS:
    paired.setdefault(record["item_id"],{})[record["model_role"]] = record
with (OUT / "annotation-table.csv").open("x",newline="") as handle:
    fields = ["item_id","question","gold_answer","full_response","retain90_response",
              "full_correctness","retain90_correctness","full_behavior","retain90_behavior","notes"]
    writer = csv.DictWriter(handle,fieldnames=fields)
    writer.writeheader()
    for fixture in fixtures:
        pair = paired[fixture["item_id"]]
        writer.writerow(dict(item_id=fixture["item_id"],question=fixture["question"],gold_answer=fixture["gold_answer"],
            full_response=pair["full"]["generated_text"],retain90_response=pair["retain90"]["generated_text"]))
print("Row 0 full:",paired[fixtures[0]["item_id"]]["full"]["generated_text"])
print("Row 0 retain:",paired[fixtures[0]["item_id"]]["retain90"]["generated_text"])
bundle = shutil.make_archive(str(OUT),"zip",root_dir=OUT)
print("Download and preserve:",bundle)
try:
    from google.colab import files
    files.download(bundle)
except ImportError:
    print("Download the ZIP from the Kaggle file browser.")
